⏱️ **Time required:** ~2 minutes | **Type:** Advanced Operation (run all cells)

# ⏪ Backfill & Replay (Time Travel)

**Owner:** Data Engineering / Platform Team  

This notebook demonstrates how to handle **backfills** and **pipeline replays** across the LakeLogic Data Mesh. Since everything is based on Delta Lake and declarative pipelines, you can easily "time travel" or replay data from raw Landing sources if a transformation logic changes.

---
## Step 1 · Environment Setup

In [ ]:
# Select execution engine (Colab comes with PySpark pre-installed)
ENGINE = "duckdb"  # 'polars' , 'spark'

In [ ]:
import os
import sys
from pathlib import Path
import polars as pl
import lakelogic as ll

PROJECT_ROOT = Path(".").resolve()
LAKEHOUSE = PROJECT_ROOT / "lakehouse"
ENV = "local"

from lakelogic.core.registry import DomainRegistry
from lakelogic.pipeline.runner import LakehousePipeline

registry = DomainRegistry.from_yaml(
    str(PROJECT_ROOT / "assets" / "domains_rideflow" / "marketplace" / "rideflow" / "_system.yaml")
)

runner = LakehousePipeline(registry, engine=ENGINE)

### ⚙️ Execution Engine

---
## Step 2 · Scenario: Re-processing Bronze Data

Imagine our Silver transformation logic changed (e.g., a bug was fixed in the metric calculation) and we need to replay our Bronze data to overwrite the Silver layer entirely.

In [ ]:
print("⏪ Forcing a Full Recompute of Silver Trips from Bronze...")

# Normally, the pipeline runs incrementally using Watermarks (Pipeline Logs).
# We can use `full_refresh=True` to ignore watermarks and replay all history.
summary = runner.run(target_layers="silver", environment=ENV, full_refresh=True)

print(summary)

---
## Step 3 · Delta Lake Time Travel (Rollback)

LakeLogic natively utilizes Delta Lake under the hood. If a pipeline run corrupts the data, you can use Delta's Time Travel capabilities to restore a previous table version.

In [ ]:
from deltalake import DeltaTable

trips_path = LAKEHOUSE / "marketplace" / "silver" / "silver_rideflow_trips"

if trips_path.exists():
    dt = DeltaTable(str(trips_path))
    history = dt.history()

    print("🕰️ Delta Lake History for `silver_rideflow_trips`:")
    for i, commit in enumerate(history):
        ts = commit.get("timestamp")
        op = commit.get("operation")
        metrics = commit.get("operationMetrics", {})
        parsed_rows = metrics.get("numOutputRows", "unknown")
        print(f"  v{commit['version']:02d} | {op:10s} | Rows Affected: {parsed_rows} | {ts}")

    if len(history) > 1:
        print("\n⏪ Restoring to previous version (v0)...")
        dt.restore(0)
        print("✅ Restore complete! Pipeline can now resume sequentially from v0.")
else:
    print("⚠️ Silver trips table not found.")

---
## Conclusion

Because the Data Mesh architectures builds on immutable storage and deterministic declarative pipelines, "Time Travel" and "Backfills" change from massive engineering headaches to simple `full_refresh` booleans and standard rollback features.